# كشكول التجارب: بناء أول محرك بحث دلالي (المحاضرات 22 و 23)

أهلاً بيك! هنا هنجمع كل اللي اتعلمناه عشان نبني **محرك بحث دلالي صغير ومتكامل (`Semantic Search Engine`)**.

الفكرة:
* المستخدم بيكتب أي سؤال بأسلوبه وبأي كلمات.
* محرك البحث بيقارن سؤال المستخدم بالوثائق اللي عنده، ويطلعله أكتر وثيقة قريبة في المعنى حتى لو مفيش ولا كلمة مشتركة بينهم!

يلا نبنيه خطوة بخطوة في كام سطر كود بسيط.


In [1]:
# 1. تجهيز الموديل وقاعدة المعرفة
import numpy as np
from sentence_transformers import SentenceTransformer

# بنستخدم الموديل السريع
embedder = SentenceTransformer("all-MiniLM-L6-v2")

# قاعدة المعرفة اللي هنبحث جواها
database = [
    {"id": 1, "text": "LangChain connects LLMs to external data and tools."},
    {"id": 2, "text": "Vector databases store embeddings for fast similarity search."},
    {"id": 3, "text": "Prompt engineering helps guide the model output without retraining."},
    {"id": 4, "text": "FastAPI is a modern, fast web framework for building APIs with Python."},
    {"id": 5, "text": "The solar system contains eight planets orbiting the Sun."}
]

# بنحسب متجهات الوثائق ونطبعها
texts = [d["text"] for d in database]
db_vectors = embedder.encode(texts, normalize_embeddings=True)

print(f"تم تخزين وفهرسة {len(database)} وثائق في قاعدة المعرفة!")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

تم تخزين وفهرسة 5 وثائق في قاعدة المعرفة!


### 2. دالة البحث الدلالي السحرية
هنعمل دالة بتاخد السؤال، وترجع أفضل النتائج مرتبة:


In [2]:
def search(query: str, top_k: int = 2):
    # 1. نحول سؤال المستخدم لمتجه
    q_vec = embedder.encode(query, normalize_embeddings=True)
    
    # 2. نحسب درجات التشابه
    scores = np.dot(db_vectors, q_vec)
    
    # 3. نرتب تنازلياً
    best_indices = np.argsort(scores)[::-1][:top_k]
    
    print(f"نتائج البحث عن: '{query}'")
    for rank, idx in enumerate(best_indices, 1):
        print(f"  المركز {rank} [شبه: {scores[idx]:.3f}]: {database[idx]['text']}")

# نجرب نسأل سؤال من غير ما نستخدم نفس كلمات الوثيقة!
search("How to store high dimensional vectors quickly?", top_k=2)


نتائج البحث عن: 'How to store high dimensional vectors quickly?'
  المركز 1 [شبه: 0.538]: Vector databases store embeddings for fast similarity search.
  المركز 2 [شبه: 0.090]: FastAPI is a modern, fast web framework for building APIs with Python.


### 3. تجربة استعلام تاني بالمعنى
بص هنا لما نسأل عن "بناء واجهات برمجية":


In [3]:
search("building backend web endpoints in python", top_k=2)


نتائج البحث عن: 'building backend web endpoints in python'
  المركز 1 [شبه: 0.546]: FastAPI is a modern, fast web framework for building APIs with Python.
  المركز 2 [شبه: 0.172]: LangChain connects LLMs to external data and tools.


---
### مساحتك للتجربة الحرة
جرب تسأل عن الكواكب أو الفضاء أو الذكاء الاصطناعي وشوف هل المحرك هيفهمك ويجيب الإجابة الصح؟


In [4]:
# جرب هنا بسؤالك:
# search("how many planets are there in space?")
